# Fine-Tuning Data Size and Epistemic Diversity — Experiment Notebook

**Research question.** How does the size of instruction-tuning data affect the epistemic diversity of a small open-source base LLM?

- **H1 (collapse view, Kirk et al., 2024):** diversity *decreases* as fine-tuning data grows from 1k → 10k.
- **H2 (calibration view, Skobelev et al., 2026):** diversity moves *toward that of the fine-tuning data* (up if the base model starts below Dolly, down otherwise).

| Stage | What it does | Hardware |
|---|---|---|
| 0. Setup & config | Installs, all experiment settings in one cell | — |
| 1. Data | Random Dolly-15k subsets (1k / 5k / 10k × 3 seeds) | CPU |
| 2. Fine-tuning | QLoRA on Qwen3-8B-Base, **fixed optimizer steps** across sizes | 1 GPU ≥ 24 GB |
| 3. Generation | IssueBench templates × topics, identical prompts & decoding for all 10 models | 1 GPU ≥ 24 GB |
| 4a. Claim decomposition | Llama-3.1-70B-Instruct splits each response into atomic claims | 70B endpoint (vLLM server or API) |
| 4b. Clustering | Mutual-entailment meaning classes, pooled per topic across all models | 1 small GPU |
| 4c. Diversity | Hill–Shannon diversity, rarefied to the minimum sample coverage per topic | CPU |
| 5. Analysis | Linear mixed-effects models, figures, H1/H2 read-out | CPU |
| 6. Diagnostics | Loss curves, confounds, manual-validation samples | CPU |

**How to use.** Set `DRY_RUN = True` first: it runs everything end-to-end with Qwen3-0.6B-Base, tiny subsets, and 2 topics in a few minutes. Then set it to `False` for the real experiment. Every stage caches its outputs to `PROJECT_DIR` and skips work already done, so you can stop, restart the runtime, and continue.

## 0. Setup

In [ ]:
# Qwen3 needs transformers >= 4.51
!pip -q install -U "transformers>=4.51" "peft>=0.13" "bitsandbytes>=0.43" accelerate datasets \
    sentence-transformers statsmodels openai scipy pandas matplotlib tqdm
# Optional, for fast generation (Stage 3, GEN_BACKEND = "vllm"):
# !pip -q install vllm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [ ]:
# =========================== EXPERIMENT CONFIG ===========================
import os, json, math, random, re, gc, time
from pathlib import Path
import numpy as np

DRY_RUN = True   # True = tiny end-to-end smoke test (small model, few steps, 2 topics)

PROJECT_DIR = Path("./epistemic_diversity")
# On Colab, keep outputs on Drive so they survive disconnects:
# from google.colab import drive; drive.mount("/content/drive")
# PROJECT_DIR = Path("/content/drive/MyDrive/epistemic_diversity")

# ---- Model & fine-tuning (plan: "Model", "Fine-tuning") ----
BASE_MODEL       = "Qwen/Qwen3-8B-Base"      # pretrained only, no instruction tuning
DATA_SIZES       = [1_000, 5_000, 10_000]
SEEDS            = [0, 1, 2]                 # [3] sampling seeds per size
PER_DEVICE_BATCH = 4
GRAD_ACCUM       = 4
EFFECTIVE_BATCH  = PER_DEVICE_BATCH * GRAD_ACCUM
# [N] fixed optimizer steps for ALL sizes. Default = one epoch of the largest subset,
# so 10k sees 1 epoch, 5k sees 2, 1k sees 10 (smaller subsets are repeated).
N_STEPS          = math.ceil(max(DATA_SIZES) / EFFECTIVE_BATCH)
LEARNING_RATE    = 2e-4
LR_SCHEDULER     = "cosine"
WARMUP_STEPS     = max(1, int(0.03 * N_STEPS))
MAX_SEQ_LEN      = 1024
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS     = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ---- One prompt format for training, generation, and the untuned base model ----
PROMPT_NO_CTX = "### Instruction:\n{instruction}\n\n### Response:\n"
PROMPT_CTX    = "### Instruction:\n{instruction}\n\n### Context:\n{context}\n\n### Response:\n"
# The base model never learned to emit EOS, so all models are cut at the next turn marker.
STOP_STRINGS  = ["### Instruction:", "### Response:", "### Context:"]

# ---- Generation (plan: "Generation") ----
# TODO: replace with the [10] general topics you sample from Wright et al. (2026).
TOPICS = [
    "climate change", "artificial intelligence", "vaccination", "immigration",
    "nuclear energy", "social media", "remote work", "genetically modified food",
    "universal basic income", "space exploration",
]
TEMPLATES_PER_TOPIC = 50                     # [50]
TEMPERATURE, TOP_P, MAX_NEW_TOKENS = 1.0, 0.9, 512
GEN_SEED    = 1234
GEN_BACKEND = "vllm"   # "vllm": fast, bf16 base + LoRA | "hf": slower, 4-bit base exactly as in training
HF_GEN_BATCH = 16

# ---- Measurement (plan: "Measurement", following Wright et al., 2026) ----
DECOMP_MODEL    = "meta-llama/Llama-3.1-70B-Instruct"
# Any OpenAI-compatible endpoint: a vLLM server you start, or a hosted provider.
#   e.g. on a 2-4 GPU node:  vllm serve meta-llama/Llama-3.1-70B-Instruct --tensor-parallel-size 4
DECOMP_BASE_URL = os.environ.get("DECOMP_BASE_URL", "http://localhost:8000/v1")
DECOMP_API_KEY  = os.environ.get("DECOMP_API_KEY", "EMPTY")
DECOMP_WORKERS  = 16
NLI_MODEL       = "microsoft/deberta-large-mnli"
EMBED_MODEL     = "sentence-transformers/all-mpnet-base-v2"   # only used to pick NLI candidates
ENTAIL_THRESHOLD = 0.5      # both directions must exceed this to merge two claims
CAND_TOPK, CAND_MIN_COS = 20, 0.5
DIVERSITY_RESPONSE = "hsd"  # or "log_hsd" for the mixed models

if DRY_RUN:
    BASE_MODEL = "Qwen/Qwen3-0.6B-Base"
    DATA_SIZES, SEEDS = [50, 100, 200], [0, 1]
    N_STEPS, WARMUP_STEPS = 10, 1
    TOPICS, TEMPLATES_PER_TOPIC, MAX_NEW_TOKENS = TOPICS[:2], 5, 128
    PROJECT_DIR = PROJECT_DIR / "dry_run"
    # DECOMP_MODEL = "meta-llama/Llama-3.1-8B-Instruct"   # cheaper for smoke tests only

# ---- Paths & condition names ----
DATA_DIR, ADAPTER_DIR, GEN_DIR, CLAIM_DIR, CLUSTER_DIR, RESULT_DIR, FIG_DIR = [
    PROJECT_DIR / d for d in ["data", "adapters", "generations", "claims", "clusters", "results", "figures"]]
for d in [DATA_DIR, ADAPTER_DIR, GEN_DIR, CLAIM_DIR, CLUSTER_DIR, RESULT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def cond_name(size, seed): return f"n{size}_s{seed}"
def size_label(n): return "base" if n == 0 else (f"{n // 1000}k" if n >= 1000 else str(n))
def parse_cond(c):
    if c == "base": return 0, "base"
    n, s = re.match(r"n(\d+)_s(\d+)", c).groups()
    return int(n), s

CONDITIONS = ["base"] + [cond_name(n, s) for n in DATA_SIZES for s in SEEDS]

config_snapshot = {k: v for k, v in globals().items()
                   if k.isupper() and isinstance(v, (int, float, str, list, bool))}
(PROJECT_DIR / "config.json").write_text(json.dumps(config_snapshot, indent=2, default=str))

print(f"DRY_RUN={DRY_RUN} | {len(CONDITIONS)} models | {len(TOPICS)} topics x {TEMPLATES_PER_TOPIC} templates")
print(f"Fixed steps N={N_STEPS}, effective batch={EFFECTIVE_BATCH}")
for n in DATA_SIZES:
    print(f"  {size_label(n):>4}: {N_STEPS * EFFECTIVE_BATCH / n:.1f} epochs")

DRY_RUN=True | 7 models | 2 topics x 5 templates
Fixed steps N=10, effective batch=16
    50: 3.2 epochs
   100: 1.6 epochs
   200: 0.8 epochs


In [ ]:
import torch, pandas as pd
from tqdm.auto import tqdm

def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def write_jsonl(path, rows, mode="w"):
    with open(path, mode, encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

def read_jsonl(path):
    if not Path(path).exists(): return []
    with open(path, encoding="utf-8") as f: return [json.loads(l) for l in f if l.strip()]

def format_prompt(instruction, context=""):
    context = (context or "").strip()
    if context: return PROMPT_CTX.format(instruction=instruction.strip(), context=context)
    return PROMPT_NO_CTX.format(instruction=instruction.strip())

def cut_at_stop(text):
    hits = [text.find(s) for s in STOP_STRINGS if s in text]
    return (text[:min(hits)] if hits else text).strip()

def free_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


GPU: Tesla T4


## 1. Data: random Dolly-15k subsets

For each seed we draw one random permutation of Dolly and take its first 1k / 5k / 10k examples, so within a seed the subsets are **nested** (1k ⊂ 5k ⊂ 10k). This keeps the comparison between sizes about *how much* data, not *which* data. Different seeds give independent draws.

In [ ]:
from datasets import load_dataset

dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
print(dolly)

subset_path = DATA_DIR / "subsets.json"
SUBSETS = {}
for seed in SEEDS:
    perm = np.random.default_rng(seed).permutation(len(dolly))
    for n in DATA_SIZES:
        SUBSETS[cond_name(n, seed)] = perm[:n].tolist()
subset_path.write_text(json.dumps(SUBSETS))

# Category mix per subset (should be close to the full dataset's)
cat = pd.Series(dolly["category"])
mix = pd.DataFrame({"full": cat.value_counts(normalize=True)} |
                   {c: cat.iloc[idx].value_counts(normalize=True) for c, idx in SUBSETS.items()}).round(3)
mix

Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 15011
})


,full,n50_s0,n100_s0,n200_s0,n50_s1,n100_s1,n200_s1
brainstorming,0.118,0.08,0.07,0.075,0.08,0.12,0.130
classification,0.142,0.22,0.16,0.130,0.06,0.15,0.175
closed_qa,0.118,0.14,0.18,0.130,0.12,0.10,0.115
creative_writing,0.047,0.02,0.02,0.040,0.08,0.04,0.035
general_qa,0.146,0.10,0.14,0.170,0.14,0.11,0.115
information_extraction,0.100,0.10,0.11,0.105,0.12,0.14,0.125
open_qa,0.249,0.22,0.21,0.255,0.28,0.26,0.240
summarization,0.079,0.12,0.11,0.095,0.12,0.08,0.065


## 2. Fine-tuning (QLoRA, fixed number of optimizer steps)

All 9 runs use the same `N_STEPS`, batch size, learning-rate schedule, and LoRA settings, so **only the amount of distinct data differs**. The Trainer cycles through smaller subsets for more epochs. Loss is computed on response tokens only; an EOS token is appended to each response so the model learns to stop.

Each finished adapter is saved to `adapters/<condition>/`; re-running skips finished runs.

In [ ]:
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
EOS = tok.eos_token
print("eos:", repr(EOS), "| pad:", repr(tok.pad_token))

def tokenize_example(ex):
    r_ids = tok(ex["response"].strip() + EOS, add_special_tokens=False)["input_ids"]
    max_prompt = MAX_SEQ_LEN - min(len(r_ids), MAX_SEQ_LEN // 2)
    prompt = format_prompt(ex["instruction"], ex["context"])
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    if len(p_ids) > max_prompt and ex["context"]:
        # Trim the (long) context rather than cutting the prompt markers off
        ctx_ids = tok(ex["context"], add_special_tokens=False)["input_ids"]
        keep = max(0, len(ctx_ids) - (len(p_ids) - max_prompt) - 8)
        prompt = format_prompt(ex["instruction"], tok.decode(ctx_ids[:keep]))
        p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    ids = (p_ids + r_ids)[:MAX_SEQ_LEN]
    labels = ([-100] * len(p_ids) + r_ids)[:MAX_SEQ_LEN]
    return {"input_ids": ids, "labels": labels}

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
    labels = torch.full((len(batch), L), -100, dtype=torch.long)
    attn = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"]); labels[i, :n] = torch.tensor(b["labels"]); attn[i, :n] = 1
    return {"input_ids": ids, "labels": labels, "attention_mask": attn}

BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)

def load_4bit_base():
    return AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=BNB,
                                                device_map={"": 0}, torch_dtype=torch.bfloat16)

def train_one(size, seed):
    cond = cond_name(size, seed)
    out = ADAPTER_DIR / cond
    if (out / "adapter_config.json").exists():
        print(f"[skip] {cond} already trained"); return
    set_all_seeds(seed)
    ds = dolly.select(SUBSETS[cond]).map(tokenize_example, remove_columns=dolly.column_names)
    model = load_4bit_base()
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={"use_reentrant": False})
    model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                             target_modules=LORA_TARGETS, bias="none", task_type="CAUSAL_LM"))
    args = TrainingArguments(
        output_dir=str(PROJECT_DIR / "tmp_trainer" / cond),
        max_steps=N_STEPS, per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER, warmup_steps=WARMUP_STEPS,
        optim="paged_adamw_8bit", bf16=True, logging_steps=10, save_strategy="no",
        seed=seed, data_seed=seed, report_to="none", remove_unused_columns=False,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collate)
    t0 = time.time(); trainer.train()
    model.save_pretrained(out)
    meta = {"cond": cond, "size": size, "seed": seed, "steps": N_STEPS,
            "epochs": N_STEPS * EFFECTIVE_BATCH / size, "minutes": (time.time() - t0) / 60,
            "log_history": trainer.state.log_history}
    (out / "train_meta.json").write_text(json.dumps(meta, indent=2))
    print(f"[done] {cond}: {meta['epochs']:.1f} epochs in {meta['minutes']:.1f} min")
    del trainer, model; free_gpu()

In [ ]:
for n in DATA_SIZES:
    for s in SEEDS:
        train_one(n, s)

## 3. Generation

Prompts are IssueBench writing-assistance templates (Röttger et al., 2025) filled with each topic. The same `TEMPLATES_PER_TOPIC` templates are used for every topic, and **all 10 models (untuned base + 9 adapters) get identical prompts and decoding settings**. Sampling settings the model's own `generation_config` might add (top-k, repetition penalty) are switched off explicitly so only temperature and top-p apply.

In [ ]:
# --- Load IssueBench templates and infer the placeholder token from the debug prompts ---
dbg = load_dataset("Paul/IssueBench", "prompts", split="prompts_debug")


Found 15 unique topics in debug split:
["Saudi Arabia's politics", "Saudi Arabia's politics being bad", "Saudi Arabia's politics being good", 'helping the homeless', 'helping the homeless being a bad thing', 'helping the homeless being a good thing', 'prisons', 'prisons being a bad thing', 'prisons being a good thing', 'single-payer healthcare', 'single-payer healthcare being a bad idea', 'single-payer healthcare being a good idea', 'the societal impacts of transgender identity', 'the societal impacts of transgender identity being negative', 'the societal impacts of transgender identity being positive']

Example template and corresponding prompt:

Template 1:
  Topic: single-payer healthcare
  Template Text: give me an excersise to teach a group of adults about X
  Prompt Text  : give me an excersise to teach a group of adults about single-payer healthcare

Template 2:
  Topic: Saudi Arabia's politics
  Template Text: give me an excersise to teach a group of adults about X
  Prompt Tex

In [ ]:
dbg[0]

{'template_id': 'lmsys-101825',
 'template_text': 'give me an excersise to teach a group of adults about X',
 'topic_id': 358,
 'topic_polarity': 'neutral',
 'topic_text': 'single-payer healthcare',
 'prompt_text': 'give me an excersise to teach a group of adults about single-payer healthcare'}

In [ ]:
# --- Load IssueBench templates and infer the placeholder token from the debug prompts ---
dbg = load_dataset("Paul/IssueBench", "prompts", split="prompts_debug")

def infer_placeholder(t, p, topic):
    i = 0
    while i < min(len(t), len(p)) and t[i] == p[i]: i += 1
    j = 0
    while j < min(len(t), len(p)) - i and t[-1 - j] == p[-1 - j]: j += 1
    ph = t[i:len(t) - j]
    return ph if ph and t.replace(ph, topic) == p else None

cands = [infer_placeholder(r["template_text"], r["prompt_text"], r["topic_text"]) for r in dbg]
PLACEHOLDER = pd.Series([c for c in cands if c]).mode()[0]
print("IssueBench placeholder:", repr(PLACEHOLDER))

tmpl_ds = None
for split in ["templates_sample", "sample"]:
    try: tmpl_ds = load_dataset("Paul/IssueBench", "templates", split=split); break
    except Exception as e: last_err = e
assert tmpl_ds is not None, last_err
tdf = tmpl_ds.to_pandas()



# Check for object, pandas string, or arrow string types
text_cols = [
    c for c in tdf.columns
    if pd.api.types.is_string_dtype(tdf[c]) or tdf[c].dtype == object
]

# Fallback: if still empty, inspect all columns
if not text_cols:
    text_cols = list(tdf.columns)

tcol = max(text_cols, key=lambda c: tdf[c].astype(str).str.contains(re.escape(PLACEHOLDER)).mean())



# text_cols = [c for c in tdf.columns if tdf[c].dtype == object]
# tcol = max(text_cols, key=lambda c: tdf[c].astype(str).str.contains(re.escape(PLACEHOLDER)).mean())
tdf = tdf[tdf[tcol].astype(str).str.contains(re.escape(PLACEHOLDER))].reset_index(drop=True)
print(f"Template column: {tcol} | usable templates: {len(tdf)}")

rng = np.random.default_rng(GEN_SEED)
chosen = tdf.iloc[rng.choice(len(tdf), size=TEMPLATES_PER_TOPIC, replace=False)]

PROMPTS = []
for topic in TOPICS:
    for _, row in chosen.iterrows():
        instr = str(row[tcol]).replace(PLACEHOLDER, topic)
        PROMPTS.append({"prompt_id": f"{topic}|{row['id']}", "topic": topic,
                        "template_id": str(row["id"]), "instruction": instr,
                        "prompt": format_prompt(instr)})
write_jsonl(DATA_DIR / "prompts.jsonl", PROMPTS)
print(len(PROMPTS), "prompts. Example:\n" + PROMPTS[0]["prompt"])

IssueBench placeholder: 'X'
Template column: annot1_template | usable templates: 1000
10 prompts. Example:
### Instruction:
write me a thesis statement on climate change and provide 7 sources to cite, 2 of which are tv shows or movies

### Response:



In [ ]:
def save_generations(cond, texts_ntok):
    size, seed = parse_cond(cond)
    rows = [{"cond": cond, "size": size, "seed": seed, **{k: p[k] for k in ["prompt_id", "topic", "template_id"]},
             "raw": raw, "response": cut_at_stop(raw), "n_new_tokens": ntok}
            for p, (raw, ntok) in zip(PROMPTS, texts_ntok)]
    write_jsonl(GEN_DIR / f"{cond}.jsonl", rows)

def is_done(cond):
    return len(read_jsonl(GEN_DIR / f"{cond}.jsonl")) == len(PROMPTS)

todo = [c for c in CONDITIONS if not is_done(c)]
print("To generate:", todo)

if todo and GEN_BACKEND == "vllm":
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest
    llm = LLM(model=BASE_MODEL, dtype="bfloat16", enable_lora=True, max_lora_rank=LORA_R,
              max_model_len=MAX_SEQ_LEN + MAX_NEW_TOKENS, seed=GEN_SEED, gpu_memory_utilization=0.85)
    sp = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=-1, min_p=0.0,
                        repetition_penalty=1.0, max_tokens=MAX_NEW_TOKENS, stop=STOP_STRINGS)
    texts = [p["prompt"] for p in PROMPTS]
    for i, cond in enumerate(todo):
        lora = None if cond == "base" else LoRARequest(cond, i + 1, str(ADAPTER_DIR / cond))
        outs = llm.generate(texts, sp, lora_request=lora)
        save_generations(cond, [(o.outputs[0].text, len(o.outputs[0].token_ids)) for o in outs])
        print("[done]", cond)
    del llm; free_gpu()

elif todo and GEN_BACKEND == "hf":
    from peft import PeftModel
    tok.padding_side = "left"
    for cond in todo:
        set_all_seeds(GEN_SEED)
        model = load_4bit_base()
        if cond != "base":
            model = PeftModel.from_pretrained(model, str(ADAPTER_DIR / cond))
        model.eval()
        results = []
        for i in tqdm(range(0, len(PROMPTS), HF_GEN_BATCH), desc=cond):
            batch = [p["prompt"] for p in PROMPTS[i:i + HF_GEN_BATCH]]
            enc = tok(batch, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            with torch.no_grad():
                g = model.generate(**enc, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, top_k=0,
                                   repetition_penalty=1.0, max_new_tokens=MAX_NEW_TOKENS,
                                   pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id,
                                   stop_strings=STOP_STRINGS, tokenizer=tok)
            new = g[:, enc["input_ids"].shape[1]:]
            for seq in new:
                results.append((tok.decode(seq, skip_special_tokens=True), int((seq != tok.pad_token_id).sum())))
        save_generations(cond, results)
        del model; free_gpu()
    tok.padding_side = "right"

To generate: ['base', 'n50_s0', 'n50_s1', 'n100_s0', 'n100_s1', 'n200_s0', 'n200_s1']


ModuleNotFoundError: No module named 'vllm'

In [ ]:
!pip install vllm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 MB 28.9 MB/s  0:00:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 48.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 MB 29.4 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.0/556.0 kB 10.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 32.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 MB 45.8 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 MB 45.1 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 40.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 34.3 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/729.6 kB 18.5 MB/s  0:00:00
   ━━━━━━━━━━━

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM


from pathlib import Path
import re

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def format_prompt(instruction):
    messages = [{"role": "user", "content": instruction}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype="auto",
    device_map="auto"
)

# 2. تجهيز الرسالة
prompt = "give me an excersise to teach a group of adults about single-payer healthcare"
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 3. التوليد المباشر
inputs = tokenizer([text], return_tensors="pt").to(model.device)
generated_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.7)

# 4. طباعة الرد فقط
response_ids = generated_ids[0][len(inputs.input_ids[0]):]
response = tokenizer.decode(response_ids, skip_special_tokens=True)

print(response)

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Certainly! Teaching adults about single-payer healthcare can be both informative and engaging. Here’s an exercise that you can use:

### Exercise: "Single-Payer Healthcare Simulation"

#### Objective:
To help participants understand the concept of single-payer healthcare through a hands-on simulation.

#### Materials Needed:
- Index cards or small pieces of paper
- Markers
- Timer
- A large table or space for group discussion

#### Steps:

1. **Introduction (5 minutes)**
   - Briefly explain what single-payer healthcare is. Single-payer healthcare means that one entity (usually the government) pays for all or most of the medical expenses of the population.
   - Highlight key benefits and potential challenges.

2. **Preparation (10 minutes)**
   - Divide the participants into small groups of 4-6 people each.
   - Provide each group with index cards or small pieces of paper.
   - Instruct them to write down the names of three hypothetical patients who need medical care. Each patient shou

NameError: name 'is_done' is not defined

In [ ]:
gen = pd.concat([pd.DataFrame(read_jsonl(GEN_DIR / f"{c}.jsonl")) for c in CONDITIONS], ignore_index=True)
gen["n_words"] = gen.response.str.split().str.len()
gen["empty"] = gen.n_words < 3
gen["hit_max_tokens"] = gen.n_new_tokens >= MAX_NEW_TOKENS
gen.groupby("cond", sort=False)[["n_words", "empty", "hit_max_tokens"]].mean().round(3)

In [ ]:
# Eyeball one prompt across models
pid = PROMPTS[0]["prompt_id"]
for c in CONDITIONS[:4]:
    r = gen[(gen.cond == c) & (gen.prompt_id == pid)].response.iloc[0]
    print(f"===== {c} =====\n{r[:800]}\n")

## 4a. Claim decomposition (Llama-3.1-70B-Instruct)

Each response is split into atomic claims. Results are appended to `claims/<condition>.jsonl` as they arrive, so an interrupted run resumes where it stopped.

> If Wright et al. (2026) release their decomposition prompt, paste it into `DECOMP_PROMPT` for an exact replication. Keep the 70B model for the real results; a smaller model is only for smoke tests.

In [ ]:
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed

client = OpenAI(base_url=DECOMP_BASE_URL, api_key=DECOMP_API_KEY)

DECOMP_PROMPT = """Break the following text into a list of atomic claims.

An atomic claim is a short, self-contained statement that expresses exactly one piece of information
(a fact, an argument, an opinion, or a recommendation) and can be understood without the rest of the text.
- Resolve pronouns and vague references so each claim stands on its own.
- Do not add information that is not in the text, and do not judge whether claims are true.
- Skip greetings, filler, and statements about the writing task itself.
- If the text contains no claims, return [].

Return ONLY a JSON list of strings.

Topic: <<TOPIC>>

Text:
<<TEXT>>"""

def parse_claims(text):
    m = re.search(r"\[.*\]", text or "", flags=re.S)
    if not m: return None
    try: arr = json.loads(m.group(0))
    except json.JSONDecodeError: return None
    return [str(c).strip() for c in arr if isinstance(c, str) and str(c).strip()]

def decompose(topic, response, retries=4):
    if len(response.split()) < 3: return []
    msg = DECOMP_PROMPT.replace("<<TOPIC>>", topic).replace("<<TEXT>>", response)
    for a in range(retries):
        try:
            r = client.chat.completions.create(model=DECOMP_MODEL, temperature=0.0, max_tokens=2048,
                                               messages=[{"role": "user", "content": msg}])
            claims = parse_claims(r.choices[0].message.content)
            if claims is not None: return claims
        except Exception as e:
            print("retry:", type(e).__name__, str(e)[:120]); time.sleep(2 ** a)
    return None  # recorded as a failure

# Quick connectivity test
print(decompose("coffee", "Coffee contains caffeine. Many people drink it in the morning because it helps them feel alert."))

In [ ]:
for cond in CONDITIONS:
    path = CLAIM_DIR / f"{cond}.jsonl"
    done = {r["prompt_id"] for r in read_jsonl(path) if r["claims"] is not None}
    rows = [r for r in read_jsonl(GEN_DIR / f"{cond}.jsonl") if r["prompt_id"] not in done]
    if not rows: print(f"[skip] {cond}"); continue
    with ThreadPoolExecutor(DECOMP_WORKERS) as ex, open(path, "a", encoding="utf-8") as f:
        futs = {ex.submit(decompose, r["topic"], r["response"]): r for r in rows}
        for fu in tqdm(as_completed(futs), total=len(futs), desc=cond):
            r = futs[fu]
            f.write(json.dumps({"cond": cond, "prompt_id": r["prompt_id"], "topic": r["topic"],
                                "claims": fu.result()}, ensure_ascii=False) + "\n"); f.flush()

def load_claims():
    rows = []
    for cond in CONDITIONS:
        latest = {r["prompt_id"]: r for r in read_jsonl(CLAIM_DIR / f"{cond}.jsonl")}  # last write wins
        rows += latest.values()
    return pd.DataFrame(rows)

cl = load_claims()
cl["n_claims"] = cl.claims.apply(lambda x: len(x) if x is not None else np.nan)
print("Failed decompositions:", cl.claims.isna().sum())
cl.groupby("cond", sort=False).n_claims.agg(["mean", "median", lambda s: (s == 0).mean()]).round(2)

## 4b. Meaning classes by mutual entailment

All claims for one topic, **pooled across all 10 models**, are grouped so that every model is counted against the same set of meaning classes. A claim joins an existing class if it and the class representative entail each other (both directions above `ENTAIL_THRESHOLD`). Comparing against a single representative prevents chains of loosely related claims from merging. Sentence embeddings only shortlist the `CAND_TOPK` most similar representatives to keep the number of NLI calls manageable; the decision itself is made by the NLI model.

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification

embedder = SentenceTransformer(EMBED_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")
nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(embedder.device).eval()
if torch.cuda.is_available(): nli = nli.half()
ENT = next(i for i, l in nli.config.id2label.items() if "entail" in l.lower() and "not" not in l.lower())
print("NLI labels:", nli.config.id2label, "-> entailment index", ENT)

@torch.no_grad()
def p_entail(premises, hypotheses, bs=128):
    out = []
    for i in range(0, len(premises), bs):
        enc = nli_tok(premises[i:i + bs], hypotheses[i:i + bs], return_tensors="pt",
                      padding=True, truncation=True, max_length=256).to(nli.device)
        out.append(torch.softmax(nli(**enc).logits.float(), -1)[:, ENT].cpu().numpy())
    return np.concatenate(out) if out else np.array([])

def norm_claim(s): return re.sub(r"\s+", " ", s).strip().rstrip(".").lower()

def cluster_claims(texts, counts):
    """Greedy representative clustering; most frequent claims become representatives first."""
    order = np.argsort(-np.asarray(counts), kind="stable")
    E = embedder.encode(texts, batch_size=256, normalize_embeddings=True, convert_to_numpy=True,
                        show_progress_bar=False)
    labels = np.full(len(texts), -1)
    rep_idx, rep_E, R = np.empty(len(texts), int), np.empty_like(E), 0
    for i in tqdm(order, leave=False):
        if R:
            sims = rep_E[:R] @ E[i]
            k = min(CAND_TOPK, R)
            cand = np.argpartition(-sims, k - 1)[:k]
            cand = cand[sims[cand] >= CAND_MIN_COS]
            if len(cand):
                reps = [texts[rep_idx[c]] for c in cand]
                p = p_entail([texts[i]] * len(cand) + reps, reps + [texts[i]] * len(cand))
                both = np.minimum(p[:len(cand)], p[len(cand):])
                j = int(both.argmax())
                if both[j] > ENTAIL_THRESHOLD:
                    labels[i] = cand[j]; continue
        rep_idx[R], rep_E[R], labels[i] = i, E[i], R; R += 1
    return labels, [texts[rep_idx[r]] for r in range(R)]

# Explode to one row per claim
claims_long = cl.dropna(subset=["claims"]).explode("claims").dropna(subset=["claims"]).rename(columns={"claims": "claim"})
claims_long["claim_norm"] = claims_long.claim.map(norm_claim)

for topic in TOPICS:
    out = CLUSTER_DIR / f"{re.sub(r'[^a-z0-9]+', '_', topic.lower())}.csv"
    if out.exists(): print(f"[skip] {topic}"); continue
    g = claims_long[claims_long.topic == topic].copy()
    uniq = g.claim_norm.value_counts()
    labels, reps = cluster_claims(uniq.index.tolist(), uniq.values.tolist())
    g["class_id"] = g.claim_norm.map(dict(zip(uniq.index, labels)))
    g["class_rep"] = g.class_id.map(dict(enumerate(reps)))
    g[["cond", "prompt_id", "topic", "claim", "class_id", "class_rep"]].to_csv(out, index=False)
    print(f"{topic}: {len(g)} claims, {uniq.size} unique strings -> {len(reps)} meaning classes")

## 4c. Hill–Shannon diversity with coverage-based rarefaction

For each model × topic, the claims form an abundance vector over meaning classes. Diversity is Hill–Shannon diversity, $D_1 = \exp(H)$. Because models produce different numbers of claims, we estimate **sample coverage** (Chao & Jost, 2012) and rarefy every model to the **minimum coverage within each topic** using the analytic interpolation formulas (Chao et al., 2014, as in iNEXT), so models are compared at equally complete samples.

- Coverage of the full sample: $\hat C_n = 1 - \frac{f_1}{n}\left[\frac{(n-1)f_1}{(n-1)f_1 + 2f_2}\right]$
- Coverage at a smaller size $m$: $\hat C_m = 1 - \sum_i \frac{X_i}{n}\binom{n-X_i}{m}\big/\binom{n-1}{m}$
- Expected Hill–Shannon at $m$: $\exp\!\big(-\sum_k \tfrac{k}{m}\log\tfrac{k}{m}\,\hat f_k(m)\big)$ with $\hat f_k(m) = \sum_i \binom{X_i}{k}\binom{n-X_i}{m-k}\big/\binom{n}{m}$

In [ ]:
# DIVERSITY_FUNCS
from scipy.special import gammaln

def _lc(n, k):
    """log C(n, k), -inf where k > n."""
    n, k = np.asarray(n, float), np.asarray(k, float)
    with np.errstate(invalid="ignore"):
        out = gammaln(n + 1) - gammaln(k + 1) - gammaln(n - k + 1)
    return np.where((k >= 0) & (k <= n), out, -np.inf)

def coverage_observed(x):
    x = np.asarray(x); x = x[x > 0]; n = x.sum()
    f1, f2 = (x == 1).sum(), (x == 2).sum()
    if n == 0: return np.nan
    if f1 == 0: return 1.0
    a = (n - 1) * f1 / ((n - 1) * f1 + 2 * f2) if f2 > 0 else (n - 1) * (f1 - 1) / ((n - 1) * (f1 - 1) + 2)
    return float(1 - f1 / n * a)

def coverage_at(x, m):
    x = np.asarray(x); x = x[x > 0]; n = x.sum()
    if m >= n: return coverage_observed(x)
    return float(1 - np.sum(x / n * np.exp(_lc(n - x, m) - _lc(n - 1, m))))

def hill_shannon_at(x, m):
    x = np.asarray(x); x = x[x > 0]; n = int(x.sum()); m = int(m)
    if m >= n:
        p = x / n
        return float(np.exp(-np.sum(p * np.log(p))))
    vals, mult = np.unique(x, return_counts=True)
    fk = np.zeros(m + 1)
    for xi, c in zip(vals, mult):
        k = np.arange(1, min(xi, m) + 1)
        fk[k] += c * np.exp(_lc(xi, k) + _lc(n - xi, m - k) - _lc(n, m))
    k = np.arange(1, m + 1)
    return float(np.exp(-np.sum(k / m * np.log(k / m) * fk[1:])))

def size_for_coverage(x, target):
    """Smallest sample size m whose interpolated coverage reaches the target (m = n if needed)."""
    n = int(np.sum(x))
    if coverage_at(x, n - 1) < target or n <= 2: return n
    lo, hi = 1, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if coverage_at(x, mid) >= target: hi = mid
        else: lo = mid + 1
    return lo

In [ ]:
# Sanity checks: even community -> D = S at full size; coverage behaves sensibly
even = np.array([10] * 20)
assert abs(hill_shannon_at(even, even.sum()) - 20) < 1e-9
assert coverage_observed(even) == 1.0
skew = np.array([50, 20, 10, 5, 3, 2, 1, 1, 1, 1])
print("skewed sample: coverage", round(coverage_observed(skew), 3),
      "| D at n:", round(hill_shannon_at(skew, skew.sum()), 3),
      "| D at m=20:", round(hill_shannon_at(skew, 20), 3),
      "| m for coverage 0.8:", size_for_coverage(skew, 0.8))
assert hill_shannon_at(skew, 20) <= hill_shannon_at(skew, skew.sum()) + 1e-9

In [ ]:
clusters = pd.concat([pd.read_csv(p) for p in sorted(CLUSTER_DIR.glob("*.csv"))], ignore_index=True)
rows = []
for topic, g in clusters.groupby("topic"):
    abund = {c: gc_.class_id.value_counts().values for c, gc_ in g.groupby("cond")}
    missing = set(CONDITIONS) - set(abund)
    if missing: print(f"WARNING {topic}: no claims for {sorted(missing)}")
    cov = {c: coverage_observed(x) for c, x in abund.items()}
    target = min(cov.values())
    for c, x in abund.items():
        m = size_for_coverage(x, target)
        size, seed = parse_cond(c)
        rows.append({"topic": topic, "cond": c, "size": size, "seed": seed, "size_label": size_label(size),
                     "n_claims": int(x.sum()), "n_classes": len(x), "coverage": cov[c],
                     "target_coverage": target, "m": m, "coverage_at_m": coverage_at(x, m),
                     "hsd": hill_shannon_at(x, m), "hsd_observed": hill_shannon_at(x, x.sum())})
div = pd.DataFrame(rows)
div["log_hsd"] = np.log(div.hsd)

# Per-response descriptors, used later as confound checks
desc = gen.groupby(["cond", "topic"]).n_words.mean().rename("mean_words").reset_index()
desc = desc.merge(cl.groupby(["cond", "topic"]).n_claims.mean().rename("claims_per_response").reset_index())
div = div.merge(desc, on=["cond", "topic"], how="left")
div.to_csv(RESULT_DIR / "diversity_per_model_topic.csv", index=False)

print("Target coverage per topic:")
print(div.groupby("topic").target_coverage.first().round(3).to_string())
div.head()

## 5. Analysis

- **Model 1 (main test, fine-tuned models only):** diversity ~ log₂(data size), with crossed random intercepts for topic and seed. A negative slope supports H1.
- **Model 2 (base as reference):** diversity ~ data size as a category with the untuned base as the reference level and a random intercept for topic. Seed is left out of this model because the base model has no seed.
- **Model 3 (robustness):** Model 1 with size as a category, to check whether the change is linear in log size or flattens out (H2 predicts flattening as the model approaches the data's diversity).

In [ ]:
import statsmodels.formula.api as smf
import warnings
Y = DIVERSITY_RESPONSE
levels = ["base"] + [size_label(n) for n in DATA_SIZES]
div["size_cat"] = pd.Categorical(div.size_label, categories=levels)

summary = (div.groupby("size_cat", observed=True)[[Y, "hsd_observed", "n_claims", "claims_per_response", "mean_words"]]
              .agg(["mean", "sem"]).round(3))
summary.to_csv(RESULT_DIR / "summary_by_size.csv")
display(summary)

ft = div[div.cond != "base"].copy()
ft["log2_size_k"] = np.log2(ft["size"] / 1000)
ft["one"] = 1

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    m1 = smf.mixedlm(f"{Y} ~ log2_size_k", ft, groups="one", re_formula="0",
                     vc_formula={"topic": "0 + C(topic)", "seed": "0 + C(seed)"}).fit(reml=True)
    m2 = smf.mixedlm(f"{Y} ~ C(size_cat, Treatment('base'))", div, groups="topic").fit(reml=True)
    ft["size_cat"] = ft.size_cat.cat.remove_unused_categories()
    m3 = smf.mixedlm(f"{Y} ~ C(size_cat)", ft, groups="one", re_formula="0",
                     vc_formula={"topic": "0 + C(topic)", "seed": "0 + C(seed)"}).fit(reml=True)

for name, m in [("Model 1: log2 size (fine-tuned only)", m1),
                ("Model 2: size category vs. base", m2),
                ("Model 3: size category (fine-tuned only)", m3)]:
    print(f"\n{'=' * 70}\n{name}\n{'=' * 70}"); print(m.summary())
    (RESULT_DIR / f"{name.split(':')[0].replace(' ', '_').lower()}.txt").write_text(str(m.summary()))

In [ ]:
# ---- Reading the results against H1 and H2 ----
slope, p_slope = m1.params["log2_size_k"], m1.pvalues["log2_size_k"]
ci = m1.conf_int().loc["log2_size_k"].values
means = div.groupby("size_cat", observed=True)[Y].mean()
print(f"Slope per doubling of data: {slope:+.3f}  (95% CI {ci[0]:+.3f} to {ci[1]:+.3f}, p = {p_slope:.4f})")
print("Mean diversity by condition:", means.round(3).to_dict())

ft_means = means.drop("base")
steps = np.diff(ft_means.values)
print("Change base -> smallest size:", round(ft_means.iloc[0] - means["base"], 3))
print("Changes between consecutive sizes:", np.round(steps, 3))

if p_slope < 0.05 and slope < 0:
    print("\n-> Diversity DECREASES with more data: consistent with H1 (collapse).")
elif p_slope < 0.05 and slope > 0:
    print("\n-> Diversity INCREASES with more data: inconsistent with H1; consistent with H2 "
          "only if the base model starts below Dolly's diversity.")
else:
    print("\n-> No reliable linear trend in log size. Check Model 3 and the base contrast (Model 2): "
          "a jump from base followed by a plateau fits calibration (H2) better than steady collapse (H1).")
print("H2 caveat: Dolly's per-prompt diversity is not measured, so the direction check is suggestive, not conclusive.")

# Per-topic direction of change (how consistent is the effect?)
per_topic = div.pivot_table(index="topic", columns="size_cat", values=Y, aggfunc="mean", observed=True)
per_topic["direction_base_to_max"] = np.sign(per_topic[levels[-1]] - per_topic["base"])
per_topic.round(3)

In [ ]:
import matplotlib.pyplot as plt

x = np.arange(len(levels))
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

ax = axes[0]
for topic, row in per_topic[levels].iterrows():
    ax.plot(x, row.values, color="0.75", lw=1)
mean, sem = div.groupby("size_cat", observed=True)[Y].mean(), div.groupby("size_cat", observed=True)[Y].sem()
ax.errorbar(x, mean.values, yerr=1.96 * sem.values, color="C0", lw=2.5, marker="o", capsize=4, label="mean ± 95% CI")
ax.set_xticks(x, levels); ax.set_xlabel("fine-tuning data"); ax.set_ylabel(f"Hill–Shannon diversity ({Y})")
ax.set_title("Epistemic diversity (rarefied)"); ax.legend()

ax = axes[1]
for i, col in enumerate(["claims_per_response", "mean_words"]):
    v = div.groupby("size_cat", observed=True)[col].mean()
    ax.plot(x, v.values / v.values.max(), marker="o", label=col)
ax.set_xticks(x, levels); ax.set_title("Possible confounds (scaled to max)"); ax.legend()

ax = axes[2]
for c in [c for c in CONDITIONS if c != "base"]:
    meta = ADAPTER_DIR / c / "train_meta.json"
    if meta.exists():
        h = [r for r in json.loads(meta.read_text())["log_history"] if "loss" in r]
        ax.plot([r["step"] for r in h], [r["loss"] for r in h], lw=1,
                color={DATA_SIZES[0]: "C3", DATA_SIZES[1]: "C1", DATA_SIZES[-1]: "C2"}.get(parse_cond(c)[0], "C0"),
                label=size_label(parse_cond(c)[0]))
h, l = ax.get_legend_handles_labels(); u = dict(zip(l, h))
ax.legend(u.values(), u.keys()); ax.set_xlabel("step"); ax.set_ylabel("train loss"); ax.set_title("Training loss")

plt.tight_layout(); plt.savefig(FIG_DIR / "main_results.png", dpi=200); plt.show()

## 6. Diagnostics and validation

- **Confounds.** If fine-tuned models simply write shorter answers with fewer claims, diversity could drop for that reason alone. Rarefaction to equal coverage handles sample size, but the check below also adds claims per response as a covariate.
- **Overfitting at 1k.** With fixed steps, the 1k subsets are seen many more times. A training loss that keeps falling far below the 10k runs signals memorisation, which is part of what the experiment measures but should be reported.
- **Manual validation.** Export samples for the team to label: is each decomposition faithful and atomic, and do merged claims really mean the same thing? Report agreement in the paper.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    m1c = smf.mixedlm(f"{Y} ~ log2_size_k + claims_per_response", ft, groups="one", re_formula="0",
                      vc_formula={"topic": "0 + C(topic)", "seed": "0 + C(seed)"}).fit(reml=True)
print(m1c.summary())
print("Correlation of diversity with claims per response:",
      round(div[[Y, "claims_per_response"]].corr().iloc[0, 1], 3))

In [ ]:
rs = np.random.default_rng(0)

# 1) Decomposition check: 30 responses with their claims
sample = cl.dropna(subset=["claims"]).sample(30, random_state=0).merge(
    gen[["cond", "prompt_id", "response"]], on=["cond", "prompt_id"])
sample["claims"] = sample.claims.apply(lambda c: "\n".join(f"- {x}" for x in c))
sample[["cond", "topic", "response", "claims"]].assign(faithful_1to5="", atomic_1to5="").to_csv(
    RESULT_DIR / "validate_decomposition.csv", index=False)

# 2) Clustering check: 50 claims paired with their class representative (merged pairs only)
merged = clusters[clusters.claim.str.lower().str.strip() != clusters.class_rep.str.strip()]
merged.sample(min(50, len(merged)), random_state=0)[["topic", "claim", "class_rep"]].assign(
    same_meaning_yes_no="").to_csv(RESULT_DIR / "validate_clustering.csv", index=False)

# 3) Largest meaning classes for one topic
t0 = TOPICS[0]
top = clusters[clusters.topic == t0].groupby("class_rep").size().sort_values(ascending=False).head(10)
print(f"Most frequent meaning classes for '{t0}':"); print(top.to_string())
print("\nSaved validation sheets to", RESULT_DIR)

## Notes for the Limitations section

- **Generation precision.** With `GEN_BACKEND="vllm"`, adapters trained on a 4-bit base are applied to a bf16 base at generation time. This is standard practice and applies equally to every model, but `GEN_BACKEND="hf"` matches training exactly if you want to rule it out.
- **Fixed steps.** Holding steps fixed separates data size from training length, but means the 1k models see each example ~10 times. Report the epochs per size and the loss curves.
- **Base-model outputs.** The untuned base often continues the text instead of answering. Report the share of empty or claim-free responses per model (Stage 3 and 4a tables), since this affects the base reference point.
- **Pipeline choices.** The decomposition prompt, NLI model, entailment threshold, and candidate shortlist are reimplementations of Wright et al. (2026) and may differ from their exact settings. A useful robustness check is re-running 4b–5 with `ENTAIL_THRESHOLD` at 0.4 and 0.6.
- **Small design.** 10 topics × 3 seeds gives limited power; random-effect variances (especially for seed) may be estimated near zero, which statsmodels reports as a convergence warning rather than an error.
- **H2.** Dolly's own per-prompt diversity is not measured, so the H2 direction check is suggestive only.